# 01 - Hola Mundo con Gemini + LangChain

Primer contacto con la API de Google Gemini usando LangChain.

**Objetivos:**
- Cargar la API key desde `.env`
- Instanciar el modelo `gemini-2.5-flash`
- Enviar un mensaje simple y recibir una respuesta

## 1. Cargar variables de entorno

In [ ]:
import os
from dotenv import load_dotenv

# Sube un nivel desde notebooks/ para encontrar el .env en la raíz
load_dotenv(dotenv_path="../.env")

api_key = os.getenv("GOOGLE_API_KEY")
print("API key cargada:", "✓" if api_key else "✗ No encontrada")

## 2. Instanciar el modelo Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.7, timeout=0.1, max_retries=3
)

print("Modelo listo:", llm.model)

## 3. Hola Mundo — primera llamada al modelo

In [ ]:
from langchain_core.messages import HumanMessage

mensaje = HumanMessage(content="Hola! Preséntate brevemente en español.")

respuesta = llm.invoke([mensaje])

print("Respuesta:", respuesta.content)
print("\n--- Metadata ---")
print("Modelo usado:", respuesta.response_metadata.get("model_version", "N/A"))
print("Tokens usados:", respuesta.usage_metadata)

## 4. Usando un prompt con string directo (forma corta)

In [ ]:
# LangChain también acepta un string directo
respuesta2 = llm.invoke("¿Cuál es la capital de México? Responde en una sola oración.")
print(respuesta2.content)

## 5. Prompt Templates

Las plantillas permiten reutilizar prompts con variables dinámicas.

| Clase | Cuándo usarla |
|---|---|
| `PromptTemplate` | Prompts de texto plano con variables |
| `ChatPromptTemplate` | Conversaciones con roles (system / human / ai) |

### 5a. PromptTemplate — texto plano

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "Explica {tema} en máximo {oraciones} oraciones, en español."
)

# Renderiza el prompt con valores concretos
prompt_listo = template.format(tema="la fotosíntesis", oraciones=2)
print(prompt_listo)

respuesta3 = llm.invoke(prompt_listo)
print("\nRespuesta:", respuesta3.content)

### 5b. ChatPromptTemplate — con roles (few-shot)

Incluye un ejemplo de entrada/salida antes de la pregunta real para guiar al modelo.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a geography expert that returns the colors present in a country's flag."),
        ("human", "France"),
        ("ai", "blue, white, red"),
        ("human", "{country}"),
    ]
)

# Genera los mensajes listos para enviar al modelo
mensajes = prompt_template.invoke({"country": "Mexico"})
print(mensajes.messages)

respuesta4 = llm.invoke(mensajes)
print("\nColores de la bandera de México:", respuesta4.content)

### 5c. FewShotChatMessagePromptTemplate — múltiples ejemplos

Cuando tienes varios ejemplos, es más limpio separarlos en una lista y usar `FewShotChatMessagePromptTemplate`. Esto te permite agregar o quitar ejemplos sin tocar la estructura del prompt.

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# Lista de ejemplos — fácil de ampliar
ejemplos = [
    {"country": "France",  "colors": "blue, white, red"},
    {"country": "Italy",   "colors": "green, white, red"},
    {"country": "Japan",   "colors": "white, red"},
    {"country": "Germany", "colors": "black, red, yellow"},
]

# Plantilla que define cómo se ve cada ejemplo
ejemplo_template = ChatPromptTemplate.from_messages([
    ("human", "{country}"),
    ("ai",    "{colors}"),
])

# Ensambla todos los ejemplos automáticamente
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=ejemplo_template,
    examples=ejemplos,
)

# Prompt final: system + ejemplos + pregunta real
prompt_final = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert that returns the colors present in a country's flag."),
    few_shot_prompt,
    ("human", "{country}"),
])

# Prueba con un país
mensajes = prompt_final.invoke({"country": "Brazil"})

# Muestra cómo quedó el prompt completo
for m in mensajes.messages:
    print(f"[{m.type}] {m.content}")

print("\n--- Respuesta del modelo ---")
respuesta5 = llm.invoke(mensajes)
print(respuesta5.content)

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

examples = [
    {
        "question": "How many Datacamp courses has Jack completed?",
        "answer": "36"
    },
    {
        "question": "How much XP does Jack have on Datacamp?",
        "answer": "284,320XP"
    },
    {
        "question": "What technology does Jack learn about most on Datacamp?",
        "answer": "Python"
    }
]
example_prompt = PromptTemplate.from_template("Question: {question}\n{answer}")
# Create the few-shot prompt
prompt_template = FewShotPromptTemplate(
      examples=examples,
      example_prompt=example_prompt,
      suffix="Question: {input}\nAnswer:",
      input_variables=["input"],
  )

# prompt = prompt_template.invoke({"input": "What is Jack's favorite technology on DataCamp?"})

llm_chain = prompt_template | llm
print(llm_chain.invoke({"input": "What is Jack's favorite technology on DataCamp?"}))


In [ ]:
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from typing import Optional


# Define the Pydantic model


class Opinion(BaseModel):
    topic: str = Field(description="The topic of the opinion")
    sentiment: str = Field(description="The sentiment of the opinion")
    problem: Optional[str] = Field(description="The problem of the opinion, if any.")
    suggested_solution: Optional[str] = Field(
        description="The suggested solution of the opinion, if any."
    )


class StructuredReview(BaseModel):
    review_id: str = Field(description="The ID of the review", pattern=r"^R\d{5}$")
    overall_sentiment: str = Field(description="The sentiment of the review")
    notable_phrases: list[str] = Field(description="The notable phrases of the review")
    opinions: list[Opinion] = Field(description="The opinions of the review")


In [ ]:
# Setup
from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.7,
)
from langchain_core.prompts import PromptTemplate


def get_response(prompt_template_str, review):
    prompt_template = PromptTemplate.from_template(prompt_template_str)
    prompt = prompt_template.format(review=review)
    response = llm.invoke(prompt)
    return response.text


# Define the Prompt Template
prompt_template_str = """
Given the following review:
{review}
Extract the following attributes from this review:
- The overall sentiment of the review (positive, negative, mixed)
- Notable phrases that capture key feedback
- Individual opinions on specific topics, each with:
  - The topic being discussed
  - The sentiment toward that topic
  - Any problem reported
  - Any solution suggested
"""

# Define the review string
review_str = """
{
    "review_id": "R12345",
    "date": "2025-01-01",
    "rating": "★★★☆☆ (3 stars)",
    "text": "I love the discount program in this app - saved 30% on my last order! However, the search functionality is really frustrating. Results are rarely relevant to what I'm looking for. They should implement category filters and improve their search algorithm."
}
"""

# Get the model's response for the review
# response_content = get_response(prompt_template_str, review_str)
# print(response_content)

# Create a structured LLM that returns validated Pydantic objects
structured_llm = llm.with_structured_output(StructuredReview)


review_text = """
review_id: R12345
date: 2025-01-01
rating: ★★★☆☆ (3 stars)
text: I love the discount program in this app - saved 30% on my last order! However, the search functionality is really frustrating. Results are rarely relevant to what I’m looking for. They should implement category filters and improve their search algorithm.
"""

prompt = f"Extract structured data from this review:\n{review_text}"

# Invoke and get a validated Pydantic object directly
structured_review = structured_llm.invoke(prompt)

print(structured_review)

# Access fields directly
print(f"\nReview ID: {structured_review.review_id}")
print(f"Sentiment: {structured_review.overall_sentiment}")
print(f"Number of opinions: {len(structured_review.opinions)}")



In [ ]:
from pydantic import BaseModel, Field
import datetime
from devtools import pprint


class Review(BaseModel):
    review_id: str = Field(description="The ID of the review", pattern=r"^R\d{5}$")
    rating: str = Field(
        description="The rating of the review", pattern=r"^★{1,5}☆{0,4} \(\d stars\)$"
    )
    date: datetime.date = Field(
        description="The date of the review", format="YYYY-MM-DD"
    )
    text: str = Field(
        description="The text of the review", min_length=1, max_length=1000
    )


invalid_review_json = """
{
  "review_id": "",
  "rating": 3,
  "date": "1 month ago",
  "text": ""
}
"""


valid_review_json = """
{
  "review_id": "R12345",
  "rating": "★★★★☆ (4 stars)",
  "date": "2025-01-01",
  "text": "This is a great product!"
}
"""

review = Review.model_validate_json(valid_review_json)

pprint(review)

